# Semana 2: Preprocesamiento de Texto
## Notebook Conceptual (NB1) – Manipulación de Texto Dummy

**Propósito:** Dominar las técnicas de limpieza y normalización de texto, incluyendo tokenización a nivel de palabra y subpalabra, stemming, lematización y uso de regex.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Implementar tokenización manual con Python (split, expresiones regulares).
- Comparar tokenizadores de NLTK, spaCy y Hugging Face.
- Aplicar stemming con PorterStemmer y SnowballStemmer.
- Aplicar lematización con WordNetLemmatizer y spaCy.
- Limpiar texto usando expresiones regulares (URLs, menciones, caracteres especiales).
- Introducir tokenización por subpalabras (BPE, WordPiece).

---

## 0. Configuración Inicial

Importamos las librerías necesarias y descargamos los recursos lingüísticos requeridos.

In [ ]:
# Importamos librerías
import re
import nltk
import spacy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# NLTK recursos
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from nltk.corpus import stopwords

# spaCy
try:
    nlp = spacy.load('en_core_web_sm')
    print("Modelo de spaCy cargado correctamente.")
except:
    print("Descargando modelo de spaCy...")
    import subprocess
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')

# Hugging Face Transformers
try:
    from transformers import AutoTokenizer
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("Nota: transformers no está instalado. Se instalará más adelante.")

# Configuración de visualización
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

print("\nLibrerías importadas correctamente.")

---
## 1. Texto Dummy con Elementos Ruidosos

Creamos un texto inventado que contiene diversos elementos que suelen aparecer en datos reales: URLs, emojis, números, mayúsculas, palabras con prefijos/sufijos, etc.

In [ ]:
texto_dummy = """
I LOOOOVE Natural Language Processing!!! 😊😊😊 Check out https://www.nlp.com/blog for more info. 
Follow me @nlp_expert #NLP #MachineLearning. The results were 100% accurate! 
I'm running, running, and running to finish this project. The children are playing happily. 
The cats are sleeping. The cat is sleeping. Studies show that studying helps.
Unbelievably, the unbelievably unbelievable event occurred.
"""

print("=== TEXTO DUMMY ===")
print(texto_dummy)

---
## 2. Tokenización Manual

### 2.1. Tokenización simple con split()

In [ ]:
# Tokenización simple por espacios
tokens_split = texto_dummy.split()
print(f"Número de tokens con split(): {len(tokens_split)}")
print("Primeros 15 tokens:")
print(tokens_split[:15])

### 2.2. Tokenización con expresiones regulares

Usamos regex para capturar palabras (incluyendo contracciones) y puntuación por separado.

In [ ]:
# Patrón para palabras (incluye contracciones) y puntuación
pattern = r"\w+(?:'\w+)?|[^\w\s]"
tokens_regex = re.findall(pattern, texto_dummy)

print(f"Número de tokens con regex: {len(tokens_regex)}")
print("Primeros 15 tokens:")
print(tokens_regex[:15])

# Mostrar la diferencia
print("\nComparación:")
print(f"split() mantiene la puntuación pegada: {tokens_split[4]} vs regex separa: {tokens_regex[4]}")

---
## 3. Tokenización con Librerías

### 3.1. NLTK word_tokenize

In [ ]:
tokens_nltk = word_tokenize(texto_dummy)
print(f"NLTK - Número de tokens: {len(tokens_nltk)}")
print("Primeros 15 tokens:")
print(tokens_nltk[:15])

### 3.2. spaCy tokenizer

In [ ]:
doc = nlp(texto_dummy)
tokens_spacy = [token.text for token in doc]
print(f"spaCy - Número de tokens: {len(tokens_spacy)}")
print("Primeros 15 tokens:")
print(tokens_spacy[:15])

### 3.3. Hugging Face BERT tokenizer (subpalabras)

Primero instalamos transformers si es necesario.

In [ ]:
if not TRANSFORMERS_AVAILABLE:
    !pip install transformers
    from transformers import AutoTokenizer

# Cargar tokenizer de BERT
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Tokenizar
tokens_bert = bert_tokenizer.tokenize(texto_dummy)
print(f"BERT tokenizer - Número de tokens: {len(tokens_bert)}")
print("Primeros 30 tokens:")
print(tokens_bert[:30])

### 3.4. Comparación de tokenizadores

In [ ]:
comparacion = pd.DataFrame({
    'Método': ['split()', 'Regex', 'NLTK', 'spaCy', 'BERT'],
    'Número de tokens': [len(tokens_split), len(tokens_regex), len(tokens_nltk), 
                         len(tokens_spacy), len(tokens_bert)]
})
print("=== Comparación de tokenizadores ===")
comparacion

---
## 4. Stemming

### 4.1. Porter Stemmer

In [ ]:
porter = PorterStemmer()

# Seleccionamos algunas palabras para stemmizar
test_words = ['running', 'runner', 'ran', 'studies', 'studying', 
              'happily', 'sleeping', 'unbelievably', 'children', 'cats']

print("=== Porter Stemmer ===")
print(f"{'Palabra':15} {'Stem':15}")
print("-" * 30)
for word in test_words:
    stem = porter.stem(word)
    print(f"{word:15} {stem:15}")

### 4.2. Snowball Stemmer

In [ ]:
snowball = SnowballStemmer('english')

print("\n=== Snowball Stemmer ===")
print(f"{'Palabra':15} {'Stem':15}")
print("-" * 30)
for word in test_words:
    stem = snowball.stem(word)
    print(f"{word:15} {stem:15}")

---
## 5. Lematización

### 5.1. WordNet Lemmatizer (NLTK)

In [ ]:
lemmatizer = WordNetLemmatizer()

print("=== WordNet Lemmatizer ===")
print(f"{'Palabra':15} {'Lema (sin POS)':20}")
print("-" * 35)
for word in test_words:
    lemma = lemmatizer.lemmatize(word)
    print(f"{word:15} {lemma:20}")

# Importancia del POS tag
print("\nCon POS tag:")
print(f"'studies' como verbo: {lemmatizer.lemmatize('studies', pos='v')}")
print(f"'studies' como sustantivo: {lemmatizer.lemmatize('studies', pos='n')}")

### 5.2. Lematización con spaCy

In [ ]:
doc_test = nlp(' '.join(test_words))

print("=== spaCy Lemmatization ===")
print(f"{'Palabra':15} {'Lema':15} {'POS'}")
print("-" * 35)
for token in doc_test:
    print(f"{token.text:15} {token.lemma_:15} {token.pos_}")

### 5.3. Comparación Stemming vs Lematización

In [ ]:
comparacion_morf = pd.DataFrame({
    'Palabra': test_words,
    'Porter Stem': [porter.stem(w) for w in test_words],
    'Snowball Stem': [snowball.stem(w) for w in test_words],
    'WordNet Lema': [lemmatizer.lemmatize(w) for w in test_words]
})
print("=== Comparación Stemming vs Lematización ===")
comparacion_morf

---
## 6. Limpieza de Texto con Expresiones Regulares

Aplicamos diversas operaciones de limpieza al texto dummy.

In [ ]:
texto_sucio = texto_dummy
print("=== TEXTO ORIGINAL ===")
print(texto_sucio)

# 1. Eliminar URLs
texto_clean = re.sub(r'https?://\S+|www\.\S+', '<URL>', texto_sucio)

# 2. Eliminar menciones (@usuario)
texto_clean = re.sub(r'@\S+', '<USER>', texto_clean)

# 3. Eliminar hashtags
texto_clean = re.sub(r'#\S+', '<HASHTAG>', texto_clean)

# 4. Eliminar emojis (rango Unicode de emojis)
emoji_pattern = re.compile("["
    u"\U0001F600-\U0001F64F"  # emoticonos
    u"\U0001F300-\U0001F5FF"  # símbolos y pictogramas
    u"\U0001F680-\U0001F6FF"  # símbolos de transporte
    u"\U0001F1E0-\U0001F1FF"  # banderas
    u"\U00002702-\U000027B0"
    u"\U000024C2-\U0001F251"
    "]+", flags=re.UNICODE)
texto_clean = emoji_pattern.sub('<EMOJI>', texto_clean)

# 5. Eliminar números (o reemplazar por token)
texto_clean = re.sub(r'\d+', '<NUM>', texto_clean)

# 6. Eliminar signos de puntuación (excepto espacios)
texto_clean = re.sub(r'[^\w\s]', ' ', texto_clean)

# 7. Normalizar espacios múltiples
texto_clean = re.sub(r'\s+', ' ', texto_clean).strip()

# 8. Convertir a minúsculas
texto_clean = texto_clean.lower()

print("\n=== TEXTO LIMPIO ===")
print(texto_clean)

---
## 7. Tokenización por Subpalabras (BPE y WordPiece)

### 7.1. Byte-Pair Encoding (BPE) con Hugging Face

Usamos GPT-2 que utiliza BPE.

In [ ]:
# Cargar tokenizer GPT-2 (BPE)
gpt2_tokenizer = AutoTokenizer.from_pretrained('gpt2')

# Palabras para probar
test_words_bpe = ['unbelievably', 'transformers', 'running', 'NLP', 'http://example.com']

print("=== BPE (GPT-2) ===")
for word in test_words_bpe:
    tokens = gpt2_tokenizer.tokenize(word)
    print(f"{word:20} -> {tokens}")

### 7.2. WordPiece (BERT)

Ya usamos BERT antes, mostramos explícitamente para subpalabras.

In [ ]:
print("=== WordPiece (BERT) ===")
for word in test_words_bpe:
    tokens = bert_tokenizer.tokenize(word)
    print(f"{word:20} -> {tokens}")

### 7.3. Comparación: BPE vs WordPiece

In [ ]:
comparacion_subword = pd.DataFrame({
    'Palabra': test_words_bpe,
    'BPE (GPT-2)': [gpt2_tokenizer.tokenize(w) for w in test_words_bpe],
    'WordPiece (BERT)': [bert_tokenizer.tokenize(w) for w in test_words_bpe]
})
print("=== Comparación BPE vs WordPiece ===")
comparacion_subword

---
## 8. Manejo de Palabras Fuera de Vocabulario (OOV)

Demostramos cómo la tokenización por subpalabras maneja palabras OOV.

In [ ]:
# Palabras inventadas (OOV)
oov_words = ['xylophonic', 'unimaginably', 'supercalifragilisticexpialidocious', 
             'covid19pandemic', 'nlpiscool']

print("=== Manejo de OOV ===")
print("\nCon tokenizador por palabras (simulado): todas serían <UNK>\n")

print("Con BPE (GPT-2):")
for word in oov_words:
    tokens = gpt2_tokenizer.tokenize(word)
    print(f"{word:40} -> {tokens}")

print("\nCon WordPiece (BERT):")
for word in oov_words:
    tokens = bert_tokenizer.tokenize(word)
    print(f"{word:40} -> {tokens}")

---
## 9. Pipeline Completo de Preprocesamiento

Unimos todos los pasos en una función que procesa un texto de entrada.

In [ ]:
def preprocess_pipeline(text, use_lemmatization=True, use_subword=False):
    """
    Pipeline completo de preprocesamiento.
    """
    # 1. Limpieza con regex
    text = re.sub(r'https?://\S+|www\.\S+', '<URL>', text)
    text = re.sub(r'@\S+', '<USER>', text)
    text = re.sub(r'#\S+', '<HASHTAG>', text)
    text = re.sub(r'\d+', '<NUM>', text)
    text = re.sub(r'[^\w\s<>]', ' ', text)  # mantener <> para los tokens especiales
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    
    # 2. Tokenización
    if use_subword:
        # Usar tokenizador BPE
        tokens = gpt2_tokenizer.tokenize(text)
        return tokens
    else:
        # Usar spaCy para tokenización y lematización
        doc = nlp(text)
        if use_lemmatization:
            return [token.lemma_ for token in doc]
        else:
            return [token.text for token in doc]

# Probamos el pipeline
print("Pipeline con tokenización y lematización (spaCy):")
result = preprocess_pipeline(texto_dummy, use_lemmatization=True, use_subword=False)
print(result[:30])

print("\nPipeline con subword tokenization (BPE):")
result_subword = preprocess_pipeline(texto_dummy, use_subword=True)
print(result_subword[:30])

---
## 10. Ejercicio para el Estudiante

Crea tu propio texto que incluya:
- URLs
- Menciones (@usuario)
- Hashtags (#tema)
- Emojis
- Números
- Palabras con prefijos/sufijos (ej. 'unbelievable', 'running')

Luego:
1. Aplica limpieza con regex para eliminar/reemplazar estos elementos.
2. Compara los resultados de stemming (Porter) vs lematización (spaCy).
3. Tokeniza con BERT y observa cómo divide las palabras.
4. Reflexiona sobre qué técnica sería más adecuada para un análisis de sentimiento.

In [ ]:
# Espacio para el estudiante
# mi_texto = "..."
# ...

---
## 11. Conclusiones

En este notebook hemos explorado las técnicas fundamentales de preprocesamiento de texto:

✔️ **Tokenización**:
- Manual con split y regex
- Automática con NLTK, spaCy
- Subpalabras con BERT y GPT-2 (BPE, WordPiece)

✔️ **Stemming vs Lematización**:
- Stemming: rápido, heurístico, produce stems no lingüísticos
- Lematización: preciso, requiere diccionario y contexto

✔️ **Limpieza con regex**:
- Eliminación de URLs, menciones, hashtags, emojis, números
- Normalización de espacios y minúsculas

✔️ **Manejo de OOV**:
- La tokenización por subpalabras resuelve eficazmente el problema de palabras fuera de vocabulario

**Lección clave**: El preprocesamiento es el cimiento de cualquier pipeline de NLP. La elección de técnicas depende del dominio y del modelo final. Los modelos modernos basados en Transformers ya incluyen tokenizadores de subpalabras, pero la limpieza inicial sigue siendo necesaria.

En el próximo notebook (NB2) aplicaremos estas técnicas a un corpus real de reseñas de películas.

---
**Fin del Notebook Conceptual - Semana 2 (NLP)**